# VoiceTut server for Spotter (open-source Egyptian voice)

1. **Runtime → Change runtime type → T4 GPU** (free).
2. **Runtime → Run all**.
3. Copy the link printed at the end (`https://….trycloudflare.com`) into the app: **Voice settings → Online voice (VoiceTut)**, then tap **Download voice**.

Keep this tab open while the app downloads. A free session stops after a few hours or when idle: run it again and use the new link.

Model: [mohammedaly22/VoiceTut-TTS](https://huggingface.co/mohammedaly22/VoiceTut-TTS) (Apache-2.0).

In [ ]:
# 1) the GPU (must say Tesla T4 or similar)
!nvidia-smi -L

In [ ]:
# 2) install VoiceTut (about 2-4 minutes)
!pip -q install git+https://github.com/k2-fsa/OmniVoice.git voicetut-tts fastapi uvicorn lameenc soundfile
# the server code with the Arabic spelling fixes, from the Spotter repo
!wget -q -O server.py https://raw.githubusercontent.com/mazenhosssam009-arch/Spotter/main/tool/voicetut/server.py
!head -3 server.py

In [ ]:
# 3) load the model and start the server (the first time downloads the model, ~1-2 GB)
import importlib, threading, time, uvicorn
import server
importlib.reload(server)
server.model()
threading.Thread(target=lambda: uvicorn.run(server.app(), host='0.0.0.0', port=8000, log_level='warning'),
                 daemon=True).start()
time.sleep(3)
print('server running')

In [ ]:
# 4) try a line here before using the app
import requests
from IPython.display import Audio, display
r = requests.post('http://localhost:8000/tts', json={'text': 'الراحَة خِلْصِت، جاهِز لِلسِّت اللي جاي؟', 'speaker': 'Yasmin'})
print(r.status_code, len(r.content), 'bytes')
open('try.mp3', 'wb').write(r.content)
display(Audio('try.mp3'))

In [ ]:
# 5) a public link for the phone (Cloudflare quick tunnel: free, no account)
import re, subprocess
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
log = open('tunnel.log', 'w')
subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=log, stderr=log)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        url = m.group(0)
        break
print('\n>>> Put this link in the app:', url or 'not ready, run this cell again')